## Pipeline to screen phenotype strength of a target feature in various datasets
* this pipeline is for screening per site values of the target feature versus control wells for a given dataset

In [10]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

import os
import pandas as pd
import numpy as np

from utils import bh_adjusted_critical_value, saveAsNewSheetToExistingFile
from scipy.stats import ttest_ind

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Set paths

In [11]:
########################## Project root directory and path to results ########################
home_path="/home/jupyter-mhaghigh@broadinst-ee45a/" #dgx
mito_project_root_dir=home_path+"bucket/projects/2016_08_01_RadialMitochondriaDistribution_donna/"
save_results_dir=mito_project_root_dir+"/workspace/results/"

### Set dataset specific parameters

In [4]:
import yaml

with open("ds_info.yaml") as f:
    cfg = yaml.safe_load(f)

datasets_path = cfg["datasets_path"]

# format only the paths, mutate in place
for name, params in cfg.items():
    if name == "datasets_path":
        continue
    params["profiles_path"] = params["profiles_path"].format(datasets_path=datasets_path)

# drop datasets_path and keep the rest as ds_info_dict
ds_info_dict = {k: v for k, v in cfg.items() if k != "datasets_path"}

## Filter list to significant perturbations and check scatter plot of t-values


In [7]:
overwrite_saved_results=False
cell_count_filter_enabled=False

####################################################
# target_feat="Cells_RadialDistribution_MeanFrac_mito_tubeness_16of16"
target_feat='slope'

sort_by_col="d_slope"
p_val_target_col="p_slope_std";#p_pattern_std, p_slope_std, p_pattern_std
p_val_orth_col="p_orth_std";#p_orth_std

####################################################

add_columns_to_print=['Count_Cells_avg','p_target_pattern', 'p_orth', 'p_slope',\
       'p_slope_std', 'p_pattern_std', 'p_orth_std', 't_target_pattern',\
       't_orth', 't_slope', 'd_slope', 'last_peak_ind', 'slope']

list_of_res_df={}
list_of_res_df1={}
list_of_res_df2={}

orth_bh_corrected_critical_dict={}
target_bh_corrected_critical_dict={}

filtering_stats=pd.DataFrame(columns=['raw','target feature significance','orth filter','both filters'])
for dataset,dataset_meta_hue in zip(['taorf','lincs','CDRP','jump_orf','jump_crispr','jump_compound'],\
                     ['Metadata_moa','Metadata_moa','Metadata_moa','Metadata_Symbol','Metadata_Symbol','Metadata_Symbol']):

    meta_cols=ds_info_dict[dataset]["meta_cols"]
    pert_col=ds_info_dict[dataset]["pert_col"]
    
    res_df=pd.read_csv(save_results_dir+'virtual_screen/'+dataset+"_results_pattern_aug_070624.csv")

    print('null size',res_df[res_df["t_orth"].isnull()].reset_index(drop=True).shape)
    uncorr_feats_condese=pd.read_csv(save_results_dir+'target_pattern_orth_features_lists/fibroblast_derived.csv')['orth_fs'].tolist()
    


    res_df=res_df[~res_df[sort_by_col].isnull()].reset_index(drop=True)
    filtering_stats.loc[dataset,'raw']=res_df.shape[0]
    
    n_perts=res_df[pert_col].unique().shape[0]

    if overwrite_saved_results:
        res_df.sort_values(by=sort_by_col).to_csv(save_results_dir+'/virtual_screen_not_filtered/'+\
                                                   dataset+'_raw_070624.csv',index=False)
    

    list_of_res_df[dataset]=res_df
    
    if 0:
        plt.figure()
        plt.scatter(res_df["Count_Cells_avg"],res_df["slope"])
        plt.title(dataset)        
        res_df[['p_orth','t_orth','Count_Cells_avg']].hist(bins=100,figsize=(10,5))
        
    
    if cell_count_filter_enabled:
        res_df_hcc=res_df[res_df['Count_Cells_avg']>res_df['Count_Cells_avg'].quantile(.1)].\
            reset_index(drop=True) 
    else:
        res_df_hcc=res_df.copy()

    
    corrected_critical = np.round(bh_adjusted_critical_value(res_df_hcc[p_val_target_col], fdr=0.05),5)
    orth_bh_corrected_critical = np.round(bh_adjusted_critical_value(res_df_hcc[p_val_orth_col], fdr=0.05),5)


    orth_bh_corrected_critical_dict[dataset] = orth_bh_corrected_critical
    target_bh_corrected_critical_dict[dataset] = corrected_critical
    print(dataset,'orth_bh_corrected_critical',orth_bh_corrected_critical)


    if 1:
        res_df_target_sig0=res_df_hcc[(res_df_hcc[p_val_target_col]<corrected_critical)].reset_index(drop=True)
        print(res_df_target_sig0.shape[0])
        filtering_stats.loc[dataset,'target feature significance']=res_df_target_sig0.shape[0]
    else:
        res_df_target_sig0=res_df.copy()        

    filtering_stats.loc[dataset,'orth filter']=res_df_hcc[(res_df_hcc[p_val_orth_col]>orth_bh_corrected_critical)].shape[0]
    
    list_of_cols_2save = [pert_col]+[sort_by_col, p_val_target_col, p_val_orth_col,'t_orth',\
                                                'Count_Cells_avg']+ meta_cols    
    
    
    if 1:
        res_df_target_sig0=res_df_target_sig0[(res_df_target_sig0[p_val_orth_col]>orth_bh_corrected_critical)].reset_index(drop=True)
        res_df_target_orth_filt=res_df_hcc[(res_df_hcc[p_val_orth_col]>orth_bh_corrected_critical)].\
        reset_index(drop=True).sort_values(by=sort_by_col)[list_of_cols_2save]
        res_df_target_orth_filt=res_df_target_orth_filt.loc[:, ~res_df_target_orth_filt.columns.duplicated(keep='last')]
        

        filtering_stats.loc[dataset,'both filters']=res_df_target_sig0.shape[0]
   
    
    res_df1=res_df_target_sig0.copy()
    res_df.loc[res_df[pert_col].isin(res_df1[pert_col].unique()),'status']='significant'
    res_df_2save=res_df.sort_values(by=sort_by_col)[list_of_cols_2save]
    res_df_2save = res_df_2save.loc[:, ~res_df_2save.columns.duplicated(keep='last')]
    
    res_df_2save_filt=res_df1.sort_values(by=sort_by_col)[list_of_cols_2save]
    res_df_2save_filt = res_df_2save_filt.loc[:, ~res_df_2save_filt.columns.duplicated(keep='last')]
    

    if 0:
        saveAsNewSheetToExistingFile(save_results_dir+'/virtual_screen_results_202407/'+dataset+'_screen_results.xlsx',\
                                 [res_df_2save,res_df_target_orth_filt,res_df_2save_filt],\
                                 [dataset, dataset+'_orthfilt',dataset+'_bothfilt'] ,keep_index_column=False)
        

    list_of_res_df1[dataset]=res_df_target_orth_filt
    list_of_res_df2[dataset]=res_df_2save_filt

    print(dataset,": original shape: ",res_df.shape,"   filtered shape:",res_df1.shape)


null size (4, 17)
taorf orth_bh_corrected_critical 0.03558
32
taorf : original shape:  (323, 18)    filtered shape: (3, 17)
null size (1, 23)
lincs orth_bh_corrected_critical 0.04829
1432
lincs : original shape:  (9394, 24)    filtered shape: (12, 23)
null size (1, 18)
CDRP orth_bh_corrected_critical 0.0483
5658
CDRP : original shape:  (30618, 19)    filtered shape: (143, 18)
null size (345, 16)
jump_orf orth_bh_corrected_critical 0.03828
587
jump_orf : original shape:  (14787, 17)    filtered shape: (60, 16)
null size (2, 16)
jump_crispr orth_bh_corrected_critical 0.04785
1117
jump_crispr : original shape:  (7975, 17)    filtered shape: (3, 16)
null size (2, 16)
jump_compound orth_bh_corrected_critical 0.04756
8457
jump_compound : original shape:  (115729, 17)    filtered shape: (229, 16)


In [13]:
target_bh_corrected_critical_dict

{'taorf': 0.00409,
 'lincs': 0.00761,
 'CDRP': 0.00924,
 'jump_orf': 0.00198,
 'jump_crispr': 0.007,
 'jump_compound': 0.00365}

In [15]:
orth_bh_corrected_critical_dict

{'taorf': 0.03558,
 'lincs': 0.04829,
 'CDRP': 0.0483,
 'jump_orf': 0.03828,
 'jump_crispr': 0.04785,
 'jump_compound': 0.04756}